# 9장 실습 — 통행시간 예측 모델

배차 알고리즘은 "어느 차가 가장 빨리 오는가"를 알아야 합니다.
그 값을 매번 다익스트라로 구하면 시뮬레이션이 느려집니다.
그래서 좌표와 시각만으로 소요시간을 예측하는 모델을 만듭니다. 교재 9장에 대응합니다.

이 장은 `scikit-learn` 과 `LightGBM` 이 필요합니다.

```bash
pip install -r requirements-heavy.txt
```

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect, todo
from smartmob.viz import use_korean_font

use_korean_font()

## 1. 정답을 만듭니다 (교재 9.1)

학습에 쓸 정답은 3장의 최단경로입니다.
무작위 O-D 쌍을 뽑아 시간대별 그래프에서 실제 소요시간을 구합니다.

교재는 2만 건을 씁니다. 여기서는 수업 시간에 맞춰 4천 건으로 줄였습니다.
20초쯤 걸립니다. 늘리려면 `n` 을 키우고 결과를 parquet 으로 저장해 두세요.

In [ ]:
import time

from smartmob.teaching.eta import FEATURES, TARGET, build_dataset

t0 = time.perf_counter()
df = build_dataset("hanam", n=4000, seed=0)
print(f"{len(df):,}건 만드는 데 {time.perf_counter() - t0:.1f}초")

print("특징:", FEATURES)
print("정답:", TARGET)
df.head(3).round(3)

특징에 라우팅 결과가 하나도 없는 것이 중요합니다.
직선거리, 시각, 방위각, 좌표뿐입니다. 전부 계산이 공짜입니다.

방위각을 `sin` 과 `cos` 두 개로 나눈 이유는 각도가 원 위의 값이기 때문입니다.
359도와 1도는 붙어 있는데 숫자로는 358만큼 떨어져 있습니다.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df[FEATURES], df[TARGET], test_size=0.2, random_state=42
)
print(f"학습 {len(X_train):,}건, 검증 {len(X_test):,}건")

## 2. 기준선 (교재 9.2)

모델을 만들기 전에 "아무것도 안 하면 얼마나 틀리는가"를 재 둡니다.
직선거리를 평균 속도로 나눈 값이 기준선입니다.
이보다 못한 모델은 쓸 이유가 없습니다.

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error

AVG_SPEED_KMH = 25.0
baseline_pred = X_test["straight_km"] / AVG_SPEED_KMH * 60

scores = {"기준선 (직선거리 ÷ 25km/h)": mean_absolute_error(y_test, baseline_pred)}
print(f"평균 절대오차 {scores['기준선 (직선거리 ÷ 25km/h)']:.2f}분")

## 3. 선형회귀 (교재 9.3)

In [ ]:
from sklearn.linear_model import LinearRegression

linear = LinearRegression().fit(X_train, y_train)
scores["선형회귀"] = mean_absolute_error(y_test, linear.predict(X_test))
print(f"평균 절대오차 {scores['선형회귀']:.2f}분")

for name, coef in sorted(zip(FEATURES, linear.coef_), key=lambda x: -abs(x[1]))[:4]:
    print(f"  {name:14s} {coef:+8.3f}")

## 4. 그래디언트 부스팅 (교재 9.4)

In [ ]:
import lightgbm as lgb

gbm = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31, verbose=-1)
gbm.fit(X_train, y_train)
scores["LightGBM"] = mean_absolute_error(y_test, gbm.predict(X_test))
print(f"평균 절대오차 {scores['LightGBM']:.2f}분")

In [ ]:
import pandas as pd

board = pd.DataFrame({"평균절대오차_분": scores}).round(3)
board["기준선 대비"] = (board["평균절대오차_분"] / scores["기준선 (직선거리 ÷ 25km/h)"]).round(2)
board

## 5. 무엇이 예측에 쓰였는가 (교재 9.5)

In [ ]:
import matplotlib.pyplot as plt

importance = pd.Series(gbm.feature_importances_, index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(importance.index, importance.values, color="#4C6EF5")
ax.set_xlabel("중요도")
ax.set_title("LightGBM 이 무엇을 보았는가")
plt.tight_layout();

## 6. 어디서 틀리는가 (교재 9.7)

평균 오차 한 줄로는 모델을 믿을 수 없습니다.
어떤 통행에서 크게 틀리는지 봐야 합니다.

In [ ]:
pred = gbm.predict(X_test)
error = pred - y_test

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y_test, pred, s=6, alpha=0.3, color="#4C6EF5")
lims = [0, max(y_test.max(), pred.max())]
axes[0].plot(lims, lims, "k--", lw=1)
axes[0].set_xlabel("실제 (분)")
axes[0].set_ylabel("예측 (분)")
axes[0].set_title("예측 대 실제")

axes[1].scatter(X_test["straight_km"], error, s=6, alpha=0.3, color="crimson")
axes[1].axhline(0, color="black", lw=1)
axes[1].set_xlabel("직선거리 (km)")
axes[1].set_ylabel("오차 (분)")
axes[1].set_title("거리에 따른 오차")
plt.tight_layout();

## 7. 배차에 넣어 보기

이 모델이 실제로 쓰이는 자리는 배차의 비용행렬입니다.
도로망 라우팅으로 만든 행렬과 얼마나 비슷한지 봅니다.

In [ ]:
from smartmob.data import load_road_graph
from smartmob.teaching.dispatch import (
    cost_matrix,
    cost_matrix_from_model,
    cost_matrix_from_router,
)

G = load_road_graph("hanam", modes=("drive",))
passengers = [(37.539, 127.215), (37.545, 127.200), (37.552, 127.190)]
vehicles = [(37.540, 127.210), (37.560, 127.195), (37.535, 127.225)]

truth = cost_matrix_from_router(passengers, vehicles, G)
straight = cost_matrix(passengers, vehicles)
model = cost_matrix_from_model(passengers, vehicles, gbm.predict, hour=18)

banner("비용행렬 세 가지의 평균 절대오차 (도로망 기준)")
print(f"직선거리 ÷ 25km/h  {np.abs(straight - truth).mean():.2f}분")
print(f"LightGBM          {np.abs(model - truth).mean():.2f}분")

## 8. 빈칸

### 8.1 특징 하나 추가하기

`FEATURES` 에 없는 특징을 하나 만들어 넣고 오차가 줄어드는지 봅니다.
후보는 여럿입니다. 출발지와 목적지의 위도 차이, 경도 차이, 도심으로부터의 거리 같은 것입니다.

라우팅 결과를 특징으로 쓰면 안 됩니다. 그러면 모델을 쓰는 의미가 없어집니다.

In [ ]:
my_feature_name = None      # 추가한 특징의 이름
my_feature_mae = None       # 그때의 평균 절대오차 (분)

banner("빈칸 8.1")
todo("추가한 특징", my_feature_name)
todo("그때의 오차", my_feature_mae, fmt=lambda v: f"{v:.2f}분")

### 8.2 학습 데이터를 늘리면

`build_dataset` 의 `n` 을 2배, 4배로 늘려 가며 오차가 어디서 더 이상 줄지 않는지 찾습니다.
데이터를 늘리는 것과 모델을 바꾸는 것 중 어느 쪽이 나은지 두 줄로 적습니다.

In [ ]:
saturation_n = None     # 오차가 더 이상 줄지 않기 시작하는 데이터 크기

banner("빈칸 8.2")
todo("포화 데이터 크기", saturation_n)

### 8.3 가장 크게 틀린 통행

오차가 가장 큰 통행 다섯 건을 찾아 출발지와 목적지를 지도에 찍어 봅니다.
왜 그 통행에서 크게 틀렸는지 한 줄로 적습니다. 한강과 산을 떠올려 보세요.

In [ ]:
worst_error_min = None      # 가장 큰 절대오차 (분)

banner("빈칸 8.3")
todo("가장 큰 오차", worst_error_min, fmt=lambda v: f"{v:.1f}분")

## 정리

- 정답은 3장의 최단경로로 만듭니다. 모델은 그것을 흉내 내되 훨씬 빠릅니다
- 기준선을 먼저 재 둡니다. 기준선을 못 이기는 모델은 쓸 이유가 없습니다
- 각도는 `sin` 과 `cos` 두 값으로 나눠 넣습니다
- 평균 오차가 아니라 어디서 틀리는지를 봐야 모델을 믿을 수 있습니다
- 10장 실습에서는 이 예측값으로 배차 비용행렬을 만듭니다